# WaveDispX 方法、公式与适用边界

## Goal

从方程到程序数据流解释 离散波数法（DWM）正演、主动/被动源、PSD 与 Poisson 时间模型、接收端互相关、Phase-shift、FJ/Bessel、Hankel 和独立理论频散，并明确数值分辨率、旁瓣来源和没有使用的“捷径”。此 Notebook 不运行耗时的波场正演。


## Setup

### 1. 加载示例与绘图函数


In [1]:
from pathlib import Path
from IPython.display import Image, display
import matplotlib.pyplot as plt

from wavedispx import CaseSession, WaveDispXWorkflow
from wavedispx.app_service import load_template_mapping
from wavedispx.preflight import checks_markdown, plot_preflight_review
from wavedispx.workflow_plotting import (
    plot_workflow_forward,
    plot_workflow_observations,
    plot_workflow_dispersion,
)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('请从 PassiveFW_versionstable 项目内运行本 Notebook')
    PROJECT_ROOT = PROJECT_ROOT.parent
print(f'Project: {PROJECT_ROOT}')

from wavedispx.preflight import build_preflight_review
from wavedispx.workflow_plotting import plot_receiver_layout_comparison


Project: /Volumes/lx_exFAT/PassiveFW_versionstable


## Steps

### 2. 离散波数法（DWM）正演：从层状动力刚度到 Green 函数

当前介质是假设水平分层、各向同性、线弹性。对每个圆频率 $\omega$ 和水平波数 $k$，DWM 组装分层动力刚度矩阵：

$$\mathbf D(k,\omega;V_P,V_S,\rho,Q_P,Q_S)\,\tilde{\mathbf u}(k,\omega)=\tilde{\mathbf f}(k,\omega),$$

因此波数域 Green 响应为 $\tilde{\mathbf G}=\mathbf D^{-1}$ 对相应载荷自由度的响应，再通过径向波数积分变回距离域：

$$G_n(r,\omega)=\int_0^\infty \tilde G_n(k,\omega)J_n(kr)\,k\,dk.$$

积分包含体波连续谱以及 Rayleigh/Love 极点贡献，所以输出是完整弹性波场；程序没有先算一条理论频散曲线再沿曲线人工填能量。Python 负责配置、geometry、频率/源分块和输出，C++17 负责主要离散波数法频率—距离块。

主动 Ricker 子波为

$$w(t)=\left[1-2(\pi f_0t)^2\right]\exp[-(\pi f_0t)^2],\qquad u_i(r,\omega)=G_{is}(r,\omega)W(\omega).$$

### 3. 被动噪声：空间位置、PSD 和激发时刻

噪声源位置可来自源环、源线或显式 `x_m,y_m` CSV。`stationary_gaussian` 没有离散激发时刻；在 PSD 频带内为每个源、每个窗生成独立复高斯系数，可写成

$$Q_s(f)=A_{PSD}(f)\frac{a_s(f)+ib_s(f)}{\sqrt 2},\qquad a_s,b_s\sim\mathcal N(0,1).$$

`poisson_ricker` 则有明确事件目录：

$$N_s\sim\mathrm{Poisson}(\lambda T_w),\quad \tau_n\sim U(0,T_w),\quad q_s(t)=\sum_{n=1}^{N_s}a_nw(t-\tau_n).$$

`noise.seed` 固定事件数、$\tau_n$、极性与幅值；程序保存 `noise_events.csv`。若 `independent_sources=True`，不同位置具有独立时间函数，这是常规环境噪声假设。

`ambient_sources.component='three_component'` 时，每个源生成独立全局 $F_x,F_y,F_z$。按源—接收器方位 $\theta$ 先计算 $F_r=F_x\cos\theta+F_y\sin\theta$、$F_t=-F_x\sin\theta+F_y\cos\theta$，再用 $u_r=G_{rr}F_r+G_{rz}F_z$、$u_t=G_{tt}F_t$、$u_z=G_{zr}F_r+G_{zz}F_z$ 传播并旋回 X/Y/Z。互相关时再按每个台站对方位形成 RR/TT/ZZ；这一步发生在真实混合记录上。

### 4. 有限记录混合与接收端 NCF

所有源先传播，然后在每个接收器相加：

$$u_i(t)=\sum_s G_{is}(t)*q_s(t).$$

对第 $m$ 个有限窗，接收器对互谱为 $U_i^{(m)}(f)U_j^{(m)*}(f)$；多窗平均并反变换得到

$$C_{ij}(\tau)=\mathcal F^{-1}\left\{\frac1M\sum_{m=1}^{M}U_i^{(m)}(f)U_j^{(m)*}(f)\right\}.$$

`spectral_whitening` 在互谱形成前压平单道振幅谱，`one_bit` 只保留时域符号，`none` 保留原始振幅。NCF 函数只接收混合后的 receiver traces，不接收源坐标或源编号；因此不存在逐源理想 Green 互相关捷径。


### 5. Phase-shift、FJ/Bessel 与 Hankel

对频率 $f$ 和候选相速度 $c$，令 $k=2\pi f/c$。Phase-shift 对距离道集补偿传播相位：

$$E_{PS}(f,c)=\left|\sum_r w(r)\hat C(f,r)\exp(ikr)\right|.$$

它对有限孔径线阵和未完全扩散的被动 NCF 更直接，因此是当前被动默认。FJ/Bessel 使用零阶柱对称核

$$E_J(f,c)=\left|\sum_r w(r)\hat C(f,r)J_0(kr)\right|,$$

Hankel 使用出射柱面波核 $H_0^{(1)}(kr)$（实现中采用与径向 Green 约定一致的核和归一化）。主动炮集默认 Hankel；被动 Bessel/Hankel 用作径向各向同性假设下的方法敏感性对照。

| 方法 | 最适合的输入 | 优点 | 主要风险 |
|---|---|---|---|
| Phase-shift | 线阵/径向 NCF 道集 | 相位关系直观，对非完全扩散较稳 | 有限孔径产生周期旁瓣 |
| FJ/Bessel | 近似各向同性、径向平均相关场 | 符合二维柱对称相关核 | 方位覆盖不足时能量变形 |
| Hankel | 主动出射柱面波、径向炮集 | 匹配柱面传播和主动源几何 | 近场、截断和核奇性需谨慎 |

### 6. 理论频散曲线如何得到

理论曲线独立求解分层介质的 Rayleigh/Love 特征条件，可概括为

$$F_R(\omega,k;V_P,V_S,\rho,h)=0,\qquad F_L(\omega,k;V_S,\rho,h)=0,\qquad c=\omega/k.$$

它使用与正演相同的层参数，但不读取合成记录，也不参与频散能量计算。白线是独立理论参考，黑线是从能量谱自动拾取，两者接近才说明数值结果与模型相符。

### 7. 分辨率、旁瓣与收敛

有限记录给出 $\Delta f\approx1/T$；径向采样的混叠近似满足 $k_{Ny}\approx\pi/\Delta r$，即 $f_{alias}\approx c_{min}/(2\Delta r)$。阵列孔径 $L$ 控制波数主瓣宽度 $\Delta k\sim2\pi/L$。因此：增加记录时长改善统计收敛和频率分辨率；增加源数改善方位照明；加长阵列收窄主瓣；减小道距提高无混叠频率。四者不能互相完全替代。

距离 taper 可压低截断旁瓣但会牺牲分辨率。亮纹只有同时满足理论曲线、不同成像方法、阵列参数敏感性和收敛检查时，才应解释为物理模态。


### 8. 同时查看线阵、二维面阵和理论频散


In [2]:
line_case = load_template_mapping(PROJECT_ROOT / 'examples/passive_quick.toml')
grid_case = load_template_mapping(PROJECT_ROOT / 'examples/passive_grid_quick.toml')
line_review = build_preflight_review(
    CaseSession(PROJECT_ROOT).apply_mapping(line_case)[0]
)
grid_review = build_preflight_review(
    CaseSession(PROJECT_ROOT).apply_mapping(grid_case)[0]
)

figure = plot_receiver_layout_comparison({'Line array': line_review, '2-D grid': grid_review})
display(figure)
plt.close(figure)

figure, axis = plt.subplots(figsize=(9, 5), constrained_layout=True)
for mode in range(line_review.theory.rayleigh_mps.shape[1]):
    axis.plot(
        line_review.theory.frequency_hz,
        line_review.theory.rayleigh_mps[:, mode],
        label=f'Rayleigh mode {mode}',
    )
axis.set(xlabel='Frequency (Hz)', ylabel='Phase velocity (m/s)', title='Independent layered-model theory')
axis.grid(alpha=0.25)
axis.legend()
display(figure)
plt.close(figure)


<Figure size 1400x900 with 4 Axes>

<Figure size 900x500 with 1 Axes>

## Checks

### 9. 结果解释边界与“没有使用的 tricks”

| 环节 | 当前稳定实现 | 不能直接解释为 |
|---|---|---|
| 模型 | 水平分层、各向同性、地表源/接收器 | 侧向非均匀或地形 |
| 主动 | 垂直/水平 Green 分量，Rayleigh/Love | 任意方位水平力未经旋转的面阵结果 |
| 被动 | 竖直快速路径，或物理 Fx/Fy/Fz 经五分量 Green 张量传播、有限混合记录、RR/TT/ZZ NCF | 逐源理想互相关或频散图后加能量 |
| 二维阵列 | 台站对按距离分箱、径向平均 | 方位各向异性成像 |
| 频散 | phase-shift / Bessel / Hankel 能量 | 每条亮纹都是物理模态 |

| 容易误解的做法 | 本程序是否使用 |
|---|---|
| 沿理论频散曲线人工添加能量 | 否 |
| 逐噪声源先做理想 Green 互相关再叠加 | 否 |
| 用源编号参与接收端 NCF | 否 |
| 使用理论曲线约束能量谱成像 | 否；理论只叠加显示和计算质量指标 |
| PSD shaping、whitening、one-bit、distance taper | 是；这些是显式可选处理参数，并非隐藏修饰 |

旁瓣和假频主要受孔径、道距、距离窗、噪声场方位覆盖和记录时长共同控制；增加噪声源只能改善照明，不能消除有限阵列的确定性旁瓣。


## Next Steps

主动操作见 `01_active_quickstart.ipynb`，被动真实处理见 `02_passive_quickstart.ipynb`。两本 Quickstart 已把全部可调参数集中到一个总控 Cell，并在输入审核中显示 required 参数路径、当前值、判据和修改方法；图形化文件选择仍可使用 `wavedispx-ui`。
